In [30]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/paysim_cleaned.csv')
conn = sqlite3.connect("../data/transactions.db")
df.to_sql("transactions",con = conn,if_exists="replace", index=False)
print("DB ready!")

DB ready!


In [31]:
df.columns

Index(['step', 'type', 'amount', 'sender_acc', 'sender_old_balance',
       'sender_new_balance', 'receiver_acc', 'receiver_old_balance',
       'receiver_new_balance', 'is_fraud', 'is_flagged_fraud',
       'transaction_hour', 'transaction_day_of_week', 'time_of_day',
       'customer_age', 'customer_gender', 'account_age_days', 'is_new_account',
       'device_type', 'channel', 'is_international', 'account_txn_count_30d',
       'merchant_category', 'customer_state', 'transaction_day', 'age_group'],
      dtype='object')

In [32]:
pd.read_sql("""
    select * from transactions limit 5
""",conn)

,step,type,amount,sender_acc,sender_old_balance,sender_new_balance,receiver_acc,receiver_old_balance,receiver_new_balance,is_fraud,...,account_age_days,is_new_account,device_type,channel,is_international,account_txn_count_30d,merchant_category,customer_state,transaction_day,age_group
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,...,1531,0,Mobile,Web,0,3,Electronics,Punjab,Monday,30-45
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,...,895,0,Tablet,App,0,16,Retail,Uttar Pradesh,Monday,30-45
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,...,37,1,Mobile,App,0,5,Travel,Maharashtra,Monday,30-45
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,...,52,1,Mobile,App,0,1,Gambling,Delhi,Monday,18-30
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,...,887,0,Mobile,Web,0,1,Food,Odisha,Monday,30-45


In [33]:
# Q1. How badly is the fraud detection system failing?How many actual frauds were never flagged by is_flagged_fraud?


pd.read_sql(
    """
    select 
    (select count(*) from transactions where is_fraud = 1) as total_frauds,
    count(*) as missed_fraud_count,
    round(100.0 * count(*) / (select count(*) from transactions where is_fraud = 1),2) as missed_fraud_percentage
    from transactions 
    where is_fraud = 1 and is_flagged_fraud = 0   
""",conn)

,total_frauds,missed_fraud_count,missed_fraud_percentage
0,8213,8197,99.81


**Insight:**

- The existing rule-based system (`is_flagged_fraud`) detected only **16** out of **8,213** actual fraud transactions.
- Around **99.81%** of fraudulent transactions were missed, indicating extremely poor fraud detection performance.
- This creates a strong need for an ML-based fraud detection model capable of identifying hidden fraud patterns.

In [34]:
# Q2. Which receiver accounts are potential money mules? One receiver collecting fraud money from multiple different senders

pd.read_sql(
    """
    select
    receiver_acc,
    count(distinct sender_acc) as unique_senders,
    count(*) as fraud_transactions,
    round(sum(amount),2) as total_fraud_amount
    from transactions
    where is_fraud = 1
    group by receiver_acc
    having unique_senders > 1
    order by unique_senders desc
    limit 10
""",conn)

,receiver_acc,unique_senders,fraud_transactions,total_fraud_amount
0,C967226405,2,2,"1,416,085.46"
1,C964377943,2,2,"6,014,862.39"
2,C935310781,2,2,"544,064.51"
3,C904300960,2,2,"1,427,683.17"
4,C803116137,2,2,"1,744,765.69"
5,C686334805,2,2,"629,142.78"
6,C668046170,2,2,"10,160,088.68"
7,C650699445,2,2,"408,028.65"
8,C644163395,2,2,"1,196,176.06"
9,C643624257,2,2,"3,071,716.97"


In [36]:
# Q3. What % of total money transacted was actually stolen via fraud?

pd.options.display.float_format = '{:,.2f}'.format  # for better readability

pd.read_sql(
    """
    with fraud_summary as (
        select 
        type as transaction_type,
        sum(case when is_fraud = 1 then amount else 0 end) as fraud_amount,
        sum(amount) as total_amount
        from transactions
        group by type
    )
    select 
    transaction_type,
    round(fraud_amount, 2) as fraud_amount,
    round(total_amount, 2) as total_amount,
    round(100.0 * fraud_amount / total_amount, 4) as loss_ratio_pct
    from fraud_summary
    order by loss_ratio_pct desc
""", conn)

,transaction_type,fraud_amount,total_amount,loss_ratio_pct
0,CASH_OUT,"5,989,202,243.83","394,412,995,224.49",1.52
1,TRANSFER,"6,067,213,184.01","485,291,987,263.17",1.25
2,CASH_IN,0.00,"236,367,391,912.46",0.00
3,DEBIT,0.00,"227,199,221.28",0.00
4,PAYMENT,0.00,"28,093,371,138.37",0.00


In [ ]:
# Q4. Are newly created accounts being used as fraud instruments?

pd.read_sql(
    """
    select 
    is_new_account,
    count(*) as total_txns,
    sum(is_fraud) as fraud_count,
    round(avg(account_age_days), 1) as avg_account_age,
    round(100.0 * sum(is_fraud) / count(*), 4) as fraud_rate_pct
    from transactions
    group by is_new_account
    order by is_new_account desc
""", conn)

,is_new_account,total_txns,fraud_count,avg_account_age,fraud_rate_pct
0,1,532907,7656,10.10,1.44
1,0,5829713,557,"1,332.50",0.01


### Insights
- New accounts have a fraud rate of **1.44%** vs **0.01%** for older accounts — a massive gap.
- Fraudulent new accounts are on average just **10 days old**, suggesting accounts are created specifically to commit fraud.

In [ ]:
# Q5. Which time periods in the simulation had the worst fraud outbreaks?

pd.read_sql(
    """
    select 
    case 
        when step between 1 and 248 then 'early (day 1-10)'
        when step between 249 and 496 then 'mid (day 11-20)'
        else 'late (day 21-31)'
    end as time_period,
    count(*) as total_txns,
    sum(is_fraud) as fraud_count,
    round(100.0 * sum(is_fraud) / count(*), 4) as fraud_rate_pct
    from transactions
    group by time_period
    order by fraud_rate_pct desc
""", conn)


,time_period,total_txns,fraud_count,fraud_rate_pct
0,late (day 21-31),306963,2698,0.88
1,mid (day 11-20),2862428,2724,0.10
2,early (day 1-10),3193229,2791,0.09


### Insights

- Fraud rate spikes sharply in the last 10 days of the month (0.88%) compared to the early and mid periods (0.09–0.10%), despite having the least transactions.
- End of the month is clearly the riskiest window for fraudulent activity.

In [39]:
# Q6. Zero-balance drain as a fraud signal — if we had used it as a detection rule,what would the precision have been?

pd.read_sql(
    """
    select 
    count(*) as total_zero_drain_txns,
    sum(is_fraud) as fraud_count,
    round(100.0 * sum(is_fraud) / count(*), 2) as fraud_rate_pct
    from transactions
    where sender_new_balance = 0
""", conn)

,total_zero_drain_txns,fraud_count,fraud_rate_pct
0,3609566,8053,0.22


### Insights

- Transactions where the sender’s balance became zero had a fraud rate of only **0.22%**.
- Zero-balance drain alone is not a strong fraud indicator, since most of these transactions were legitimate.

In [ ]:
# Q7 Are big transactions more likely to be fraud?

p99 = df['amount'].quantile(0.99)

pd.read_sql(
    f"""
    select 
    case 
        when amount > {p99} then 'above 99th percentile'
        else 'below 99th percentile'
    end as amount_segment,
    count(*) as total_txns,
    sum(is_fraud) as fraud_count,
    round(100.0 * sum(is_fraud) / count(*), 4) as fraud_rate_pct
    from transactions
    group by amount_segment
""", conn)

,amount_segment,total_txns,fraud_count,fraud_rate_pct
0,above 99th percentile,63627,1969,3.09
1,below 99th percentile,6298993,6244,0.10


### Insights
- Top 1% transactions are 30x more likely to be fraud (3.09% vs 0.10%).
- But 96% of even large transactions are legitimate — amount alone cannot flag fraud reliably.

In [ ]:
# Q8 Which age group is most targeted by fraudsters?

pd.read_sql(
    """
    select 
    age_group,
    count(*) as total_txns,
    sum(is_fraud) as fraud_counts ,
    round(100.0 * sum(is_fraud) / count(*),4) as fraud_rate_pct,
    round(avg(case when is_fraud = 1 then amount end),2) as avg_fraud_amount
    from transactions
    group by age_group
    order by fraud_rate_pct desc
""",conn)

,age_group,total_txns,fraud_counts,fraud_rate_pct,avg_fraud_amount
0,60+,177677,2272,1.28,"1,460,691.91"
1,45-60,1431740,3124,0.22,"1,499,091.43"
2,30-45,2972728,2176,0.07,"1,435,334.73"
3,18-30,1780475,641,0.04,"1,452,844.62"


### Insights

- Fraud rate increases sharply with age — **60+ group has the highest fraud rate at 1.28%**, nearly 32x more than 18-30.
- Older customers are the primary target, not younger ones.

In [ ]:
# Q9. Which state and channel combination sees the most fraud?

pd.read_sql(
    """
    select 
    customer_state,
    channel,
    sum(is_fraud) as fraud_counts,
    round(100.0 * sum(is_fraud) / count(*),4) as fraud_rate_pct
    from transactions
    group by customer_state,channel
    order by fraud_rate_pct desc limit 10
""",conn)

,customer_state,channel,fraud_counts,fraud_rate_pct
0,Maharashtra,ATM,385,1.20
1,Delhi,ATM,309,0.96
2,Karnataka,ATM,286,0.89
3,Tamil Nadu,ATM,234,0.73
4,Maharashtra,App,784,0.49
5,Maharashtra,POS,72,0.45
6,Delhi,App,668,0.42
7,Delhi,POS,64,0.41
8,Karnataka,POS,57,0.35
9,Karnataka,App,555,0.35


### Insights
- ATM is the riskiest channel with the highest fraud rate across all states — Maharashtra, Delhi, and Karnataka lead.
- App has higher fraud counts but ATM has higher fraud rate, meaning ATM transactions are harder to catch.

In [ ]:
# Q10. Which merchant category has the highest fraud rate and average fraud amount?

pd.read_sql(
    """
    select
    merchant_category,
    sum(is_fraud) as fraud_counts,
    round(100.0 * sum(is_fraud) / count(*),4) as fraud_rate_pct,
    round(avg(case when is_fraud = 1 then amount end),2) as avg_amount
    from transactions
    group by merchant_category
    order by fraud_rate_pct desc
""",conn)

,merchant_category,fraud_counts,fraud_rate_pct,avg_amount
0,Gambling,2280,1.18,"1,476,724.93"
1,Electronics,3110,0.33,"1,514,386.35"
2,Travel,1371,0.31,"1,367,824.50"
3,Retail,852,0.04,"1,528,289.22"
4,Food,425,0.03,"1,320,974.93"
5,Utilities,175,0.01,"1,376,782.23"


### Insights
- **Gambling** has the highest fraud rate at **1.18%** — nearly 4x higher than Electronics and Travel.
- Electronics has the most fraud cases (**3,110**) despite a lower rate — high transaction volume makes it a major fraud channel.